MedMCQA: updated manuscript analysis..


In [1]:
from pathlib import Path
import os
_REPOSITORY_ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / 'experiments/clinical_protocols').is_dir() and (p / 'data').is_dir()))
if os.name == 'nt' and (not str(_REPOSITORY_ROOT).startswith('\\\\?\\')):
    _REPOSITORY_ROOT = Path('\\\\?\\' + str(_REPOSITORY_ROOT))
_NOTEBOOK_DIR = _REPOSITORY_ROOT / 'experiments/external_benchmarks/medmcqa/benchmark_subsets'
os.chdir(_NOTEBOOK_DIR)


In [2]:
from pathlib import Path
from collections import Counter
import hashlib
import json
import math
import numpy as np
import pandas as pd
from scipy.stats import binomtest
from scipy.stats import t as student_t
from statsmodels.stats.multitest import multipletests

def locate_root():
    import os
    start = Path.cwd().resolve()
    for parent in (start, *start.parents):
        if (parent / 'data/limfmed_dataset_gold.csv').is_file() and (parent / 'experiments').is_dir():
            if os.name == 'nt' and (not str(parent).startswith(chr(92) * 2 + '?' + chr(92))):
                return Path(chr(92) * 2 + '?' + chr(92) + str(parent))
            return parent
    raise FileNotFoundError('Run the notebook from inside the repository.')
ROOT = locate_root()
OUT = ROOT / 'experiments/external_benchmarks/medmcqa/benchmark_subsets'
PREFIX = 'benchmark_condition_comparisons'
PACKAGE = OUT / 'shared_benchmark_runs'
SUBSETS = ['random_dev300', 'shao_hard100', 'goa100', 'marc100', 'marc250', 'dev4183']
NEW_RUNS = {'Delphi': PACKAGE / 'runs/primary/delphi_gpt120_qwen36_deepseekv4', 'GPT-OSS-120B': PACKAGE / 'runs/primary/baseline_gpt120', 'Qwen3.6-35B-A3B': PACKAGE / 'runs/primary/baseline_qwen36', 'DeepSeek-V4 Flash': OUT / 'deepseek_baseline_runs/runs/primary/baseline_deepseekv4'}
BASELINES = ['GPT-OSS-120B', 'Qwen3.6-35B-A3B', 'DeepSeek-V4 Flash']
IDENTITY = ['question', 'option_a', 'option_b', 'option_c', 'option_d', 'correct_option_letter']
input_provenance = []

def digest(path):
    h = hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda: f.read(2 ** 20), b''):
            h.update(chunk)
    return h.hexdigest()

def read_result(path):
    exact = set(['case_id', 'table_source', *IDENTITY, 'subject_name', 'topic_name', 'baseline_selected_option', 'delphi_consensus_selected_option', 'winner_selected_option', 'delphi_rounds_executed', 'delphi_early_stop_reason'])

    def keep(c):
        return c in exact or (c.startswith(('r1_', 'r2_', 'r3_')) and ('selected_option__' in c or 'confidence__' in c))
    d = pd.read_csv(path, sep=';', encoding='utf-8-sig', dtype=str, usecols=keep).fillna('')
    if 'table_source' in d:
        d = d.loc[d.table_source == 'result_df'].copy()
    else:
        d = d.loc[d.case_id != ''].copy()
    assert not d.case_id.duplicated().any(), f'Duplicate case IDs: {path}'
    assert (d.case_id != '').all(), f'Missing case IDs: {path}'
    input_provenance.append({'path': str(path.relative_to(ROOT)), 'sha256': digest(path), 'n': len(d)})
    return d.set_index('case_id')

def opt(x):
    s = str(x).strip().upper()[:1]
    return s if s in {'A', 'B', 'C', 'D'} else ''

def plurality(labels, organizer, confidences=None):
    valid_labels = [x for x in labels if x]
    if not valid_labels:
        return ''
    c = Counter(valid_labels)
    tied = [x for x, n in c.items() if n == max(c.values())]
    if len(tied) == 1:
        return tied[0]
    if organizer in tied:
        return organizer
    if confidences is not None:
        scores = {label: 0.0 for label in tied}
        for label, confidence in zip(labels, confidences):
            if label in scores:
                try:
                    scores[label] += float(confidence)
                except (ValueError, TypeError):
                    pass
        return max(tied, key=lambda label: scores[label])
    return tied[0]

def wilson(k, n):
    z = 1.959963984540054
    p = k / n
    den = 1 + z * z / n
    centre = (p + z * z / (2 * n)) / den
    half = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / den
    return [centre - half, centre + half]

def paired(a, b):
    a, b = (np.asarray(a, bool), np.asarray(b, bool))
    n10 = int(np.sum(a & ~b))
    n01 = int(np.sum(~a & b))
    return {'n': len(a), 'accuracy_a': float(a.mean()), 'accuracy_b': float(b.mean()), 'difference_pp': float(100 * (a.mean() - b.mean())), 'a_only_correct': n10, 'b_only_correct': n01, 'p_exact': float(binomtest(n10, n10 + n01, 0.5).pvalue) if n10 + n01 else 1.0}

def bootstrap_difference(case_differences, seed=20261011, draws=10000):
    x = np.asarray(case_differences, float)
    if not len(x):
        return [None, None]
    rng = np.random.default_rng(seed)
    means = []
    for start in range(0, draws, 100):
        means.extend(rng.choice(x, size=(min(100, draws - start), len(x)), replace=True).mean(axis=1))
    return [float(v * 100) for v in np.quantile(means, [0.025, 0.975])]

def bootstrap_cluster_ratio(numerators, denominators, seed=20261011, draws=10000):
    numerators, denominators = (np.asarray(numerators), np.asarray(denominators))
    if not denominators.sum():
        return [None, None]
    rng = np.random.default_rng(seed)
    ratios = []
    for start in range(0, draws, 100):
        idx = rng.integers(0, len(numerators), size=(min(100, draws - start), len(numerators)))
        den = denominators[idx].sum(axis=1)
        num = numerators[idx].sum(axis=1)
        ratios.extend(num[den > 0] / den[den > 0])
    return [float(v * 100) for v in np.quantile(ratios, [0.025, 0.975])]


In [3]:
new_frames = {}
accuracy_rows, comparison_rows, majority_rows, oracle_rows = ([], [], [], [])
configuration_rows, overlap_rows = ([], [])
manifest = json.loads((PACKAGE / 'data_manifest.json').read_text(encoding='utf-8'))
manifest_by_id = {x['id']: x for x in manifest['subsets']}
for subset in SUBSETS:
    frames = {system: read_result(folder / f'{subset}.csv') for system, folder in NEW_RUNS.items()}
    panel = frames['Delphi']
    order = panel.index
    for system, d in frames.items():
        assert set(order) == set(d.index), f'Different question membership: {subset}/{system}'
        d = d.reindex(order)
        assert panel[IDENTITY].equals(d[IDENTITY]), f'Question/options/gold mismatch: {subset}/{system}'
        frames[system] = d
    truth = panel.correct_option_letter.map(opt)
    assert truth.ne('').all()
    pred = {s: d['delphi_consensus_selected_option' if s == 'Delphi' else 'baseline_selected_option'].map(opt) for s, d in frames.items()}
    corr = {s: p.eq(truth) for s, p in pred.items()}
    for system in NEW_RUNS:
        k, n = (int(corr[system].sum()), len(panel))
        lo, hi = wilson(k, n)
        met = json.loads((NEW_RUNS[system] / f'{subset}.metrics.json').read_text(encoding='utf-8'))
        assert (k, n) == (met['correct'], met['n']), f'Recomputed/recorded metrics mismatch: {subset}/{system}'
        accuracy_rows.append({'subset': subset, 'system': system, 'n': n, 'correct': k, 'accuracy_pct': 100 * k / n, 'ci_low_pct': 100 * lo, 'ci_high_pct': 100 * hi, 'invalid': int(pred[system].eq('').sum())})
        cfg = met['configuration']
        configuration_rows.append({'subset': subset, 'system': system, 'temperature': cfg.get('temperature', cfg.get('temperatures')), 'model_output_limits': cfg.get('model_output_limits'), 'configuration_signature': met['configuration_signature']})
    for system in BASELINES:
        ci = bootstrap_difference(corr['Delphi'].astype(float) - corr[system].astype(float))
        comparison_rows.append({'subset': subset, 'comparator': system, **paired(corr['Delphi'], corr[system]), 'difference_ci_low_pp': ci[0], 'difference_ci_high_pp': ci[1]})
    maj = pd.Series([plurality([pred[s].loc[i] for s in BASELINES], pred['GPT-OSS-120B'].loc[i]) for i in order], index=order)
    majority_rows.append({'subset': subset, 'comparator': 'independent baseline majority (GPT breaks ties)', **paired(corr['Delphi'], maj.eq(truth))})
    oracle = pd.concat([corr[s] for s in BASELINES], axis=1).any(axis=1)
    oracle_rows.append({'subset': subset, 'comparator': 'any-correct independent-member oracle', **paired(corr['Delphi'], oracle)})
    new_frames[subset] = frames
for i, a in enumerate(SUBSETS):
    for b in SUBSETS[i + 1:]:
        overlap_rows.append({'subset_a': a, 'subset_b': b, 'shared_question_ids': len(set(new_frames[a]['Delphi'].index) & set(new_frames[b]['Delphi'].index))})
accuracy = pd.DataFrame(accuracy_rows)
comparisons = pd.DataFrame(comparison_rows)
comparisons['p_holm_three_members_within_subset'] = comparisons.groupby('subset', sort=False).p_exact.transform(lambda x: multipletests(x, method='holm')[1])
comparisons['p_holm_all_18'] = multipletests(comparisons.p_exact, method='holm')[1]
majorities = pd.DataFrame(majority_rows)
oracles = pd.DataFrame(oracle_rows)
majorities['p_holm_six_subsets'] = multipletests(majorities.p_exact, method='holm')[1]
oracles['p_holm_six_subsets'] = multipletests(oracles.p_exact, method='holm')[1]


In [4]:
round_case_records = []

def round_analysis(d, name, group, organizer):
    pids = sorted((c.split('__', 1)[1] for c in d if c.startswith('r1_selected_option__')))
    assert len(pids) == 3, (name, pids)
    truth = d.correct_option_letter.map(opt)
    r1 = d[[f'r1_selected_option__{p}' for p in pids]].map(opt)
    executed = pd.to_numeric(d.delphi_rounds_executed).astype(int)
    last = pd.DataFrame(index=d.index, columns=pids)
    for case in d.index:
        for p in pids:
            last.loc[case, p] = opt(d.loc[case].get(f'r{executed.loc[case]}_selected_option__{p}', ''))
    init_vote = pd.Series([plurality(r1.loc[i].tolist(), opt(d.loc[i, f'r1_selected_option__{organizer}']), [d.loc[i].get(f'r1_confidence__{p}', '') for p in pids]) for i in d.index], index=d.index)
    fin = d.delphi_consensus_selected_option.map(opt)
    contested = executed >= 2
    x = paired(fin[contested].eq(truth[contested]), init_vote[contested].eq(truth[contested]))
    all_x = paired(fin.eq(truth), init_vote.eq(truth))
    r1_correct = r1.eq(truth, axis=0)
    last_correct = last.eq(truth, axis=0)
    r1v, lastv = (r1_correct.to_numpy(), last_correct.to_numpy())
    mask = contested.to_numpy()
    valid_pairs = r1.ne('').to_numpy() & last.ne('').to_numpy()
    individual = paired(lastv[mask][valid_pairs[mask]], r1v[mask][valid_pairs[mask]])
    case_gain_numerators = ((lastv.astype(int) - r1v.astype(int)) * valid_pairs)[mask].sum(axis=1)
    case_gain_denominators = valid_pairs[mask].sum(axis=1)
    for case in d.index:
        j = d.index.get_loc(case)
        round_case_records.append({'group': group, 'panel': name, 'case_id': case, 'contested': bool(contested.loc[case]), 'r1_plurality_correct': bool(init_vote.loc[case] == truth.loc[case]), 'final_correct': bool(fin.loc[case] == truth.loc[case]), 'individual_gain_numerator': int(case_gain_numerators[np.sum(mask[:j])] if mask[j] else 0), 'individual_valid_pairs': int(case_gain_denominators[np.sum(mask[:j])] if mask[j] else 0)})
    row = {'group': group, 'panel': name, 'questions': len(d), 'contested': int(contested.sum()), 'r1_unanimous': int((executed == 1).sum()), 'stopped_after_r2': int((executed == 2).sum()), 'ran_r3': int((executed == 3).sum()), 'mean_rounds': float(executed.mean()), 'accuracy_all_r1_plurality_pct': 100 * all_x['accuracy_b'], 'accuracy_all_final_pct': 100 * all_x['accuracy_a'], 'accuracy_contested_r1_pct': 100 * x['accuracy_b'], 'accuracy_contested_final_pct': 100 * x['accuracy_a'], 'difference_contested_pp': x['difference_pp'], 'repaired': x['a_only_correct'], 'corrupted': x['b_only_correct'], 'p_exact': x['p_exact'], 'changed_option_contested': int(fin[contested].ne(init_vote[contested]).sum()), 'initial_oracle_accuracy_pct': 100 * r1_correct.any(axis=1).mean(), 'final_correct_when_all_r1_wrong': int((fin.eq(truth) & ~r1_correct.any(axis=1)).sum()), 'individual_contested_observations': individual['n'], 'individual_contested_r1_pct': 100 * individual['accuracy_b'], 'individual_contested_last_pct': 100 * individual['accuracy_a'], 'individual_repaired': individual['a_only_correct'], 'individual_corrupted': individual['b_only_correct'], 'individual_difference_pp': individual['difference_pp'], 'individual_case_bootstrap_ci95_pp': bootstrap_cluster_ratio(case_gain_numerators, case_gain_denominators), 'individual_missing_pair_exclusions': int((~valid_pairs[mask]).sum()), 'invalid_round1_decisions': int(r1.eq('').sum().sum()), 'invalid_last_decisions': int(last.eq('').sum().sum()), 'winner_consensus_mismatches': int(d.winner_selected_option.map(opt).ne(fin).sum()) if 'winner_selected_option' in d else None}
    influence = []
    for case in d.index[contested]:
        labels = r1.loc[case].tolist()
        c = Counter(labels)
        if '' in c or len(c) != 2 or max(c.values()) != 2:
            continue
        maj = max(c, key=c.get)
        for p, label in zip(pids, labels):
            influence.append({'case_id': case, 'position': 'majority' if label == maj else 'minority', 'majority_correct': maj == truth.loc[case], 'ended_on_initial_majority': last.loc[case, p] == maj})
    inf = pd.DataFrame(influence)
    inf_summary = []
    if not inf.empty:
        for (position, correct), g in inf.groupby(['position', 'majority_correct']):
            inf_summary.append({'group': group, 'panel': name, 'position': position, 'majority_correct': bool(correct), 'n_decisions': len(g), 'ended_on_initial_majority': int(g.ended_on_initial_majority.sum()), 'share_pct': 100 * g.ended_on_initial_majority.mean()})
    return (row, inf_summary)


In [5]:
round_rows, influence_rows = ([], [])
for subset in SUBSETS:
    row, inf = round_analysis(new_frames[subset]['Delphi'], subset, 'new', 'gpt120b_default')
    round_rows.append(row)
    influence_rows.extend(inf)
LEGACY = {'DE-7G': ('experiments/external_benchmarks/medmcqa/panel_gpt120_organizer_qwen_deepseek/delphi_medmcqa_noroles.csv', 'gpt120b_default'), 'DE-5G': ('experiments/external_benchmarks/initial_runs/heterogeneous_gpt120_organizer/delphi_medmcqa_noroles.csv', 'gpt120b_default'), 'DE-6G': ('experiments/external_benchmarks/initial_runs/panel_gpt120_organizer_qwen_alice/delphi_medmcqa_noroles.csv', 'gpt120b_default'), 'DH-D': ('experiments/external_benchmarks/initial_runs/three_deepseek_members/delphi_medmcqa_noroles.csv', 'deepseekv4_default')}
legacy_frames = {}
for name, (relative_path, organizer) in LEGACY.items():
    d = read_result(ROOT / relative_path)
    row, inf = round_analysis(d, name, 'legacy_train1000', organizer)
    round_rows.append(row)
    influence_rows.extend(inf)
    legacy_frames[name] = d
rounds = pd.DataFrame(round_rows)
rounds['p_holm_within_group'] = rounds.groupby('group', sort=False).p_exact.transform(lambda x: multipletests(x, method='holm')[1])
old_audit = pd.read_csv(ROOT / 'results/clinical_evaluation/tables/medmcqa_accuracy_by_round.csv', sep=';')
for name, alias in {'DE-7G': 'AC (gpt·qwen3-235b·deepseek-v4)', 'DE-5G': 'Optimum (gpt·qwen3-235b·qwen3.6-35b)', 'DE-6G': 'AC-Alice (gpt·qwen3-235b·alice)', 'DH-D': 'DeepSeek trio (deepseek-v4 ×3)'}.items():
    orig = old_audit.loc[old_audit.panel == alias].iloc[0]
    new = rounds.loc[rounds.panel == name].iloc[0]
    assert int(orig.contested) == int(new.contested)
    assert int(orig.fixed_by_rounds) == int(new.repaired)
    assert int(orig.broken_by_rounds) == int(new.corrupted)
    assert abs(orig.acc_contested_r1_plurality * 100 - new.accuracy_contested_r1_pct) < 1e-08


In [6]:
legacy_accuracy_rows = []
legacy_baselines = {'GPT-OSS-120B': 'experiments/external_benchmarks/medmcqa/baseline_gpt/baseline_medmcqa_gpt.csv', 'Qwen3-235B': 'experiments/external_benchmarks/medmcqa/baseline_qwen/baseline_medmcqa_qwen.csv', 'Qwen3.6-35B-A3B': 'experiments/external_benchmarks/initial_runs/baseline_qwen36/baseline_medmcqa_qwen36.csv', 'DeepSeek-V4 Flash': 'experiments/external_benchmarks/medmcqa/baseline_deepseekv4/baseline_medmcqa_deepseekv4.csv', 'Alice': 'experiments/external_benchmarks/initial_runs/baseline_alice/baseline_medmcqa_alice.csv'}
legacy_all = {**legacy_frames, **{name: read_result(ROOT / path) for name, path in legacy_baselines.items()}}
reference = legacy_frames['DE-7G']
for name, d in legacy_all.items():
    assert set(d.index) == set(reference.index)
    d = d.reindex(reference.index)
    assert reference[IDENTITY].equals(d[IDENTITY])
    col = 'delphi_consensus_selected_option' if name in LEGACY else 'baseline_selected_option'
    c = d[col].map(opt).eq(d.correct_option_letter.map(opt))
    legacy_accuracy_rows.append({'system': name, 'n': len(d), 'correct': int(c.sum()), 'accuracy_pct': 100 * c.mean()})
legacy_accuracy = pd.DataFrame(legacy_accuracy_rows)
round_cases = pd.DataFrame(round_case_records)
old_cases = round_cases.loc[(round_cases.group == 'legacy_train1000') & round_cases.contested].copy()
old_cases['gain_numerator'] = old_cases.final_correct.astype(int) - old_cases.r1_plurality_correct.astype(int)
old_clusters = old_cases.groupby('case_id').agg(gain_numerator=('gain_numerator', 'sum'), panel_case_denominator=('panel', 'size'), individual_gain_numerator=('individual_gain_numerator', 'sum'), individual_valid_pairs=('individual_valid_pairs', 'sum'))
old_pooled = paired(old_cases.final_correct, old_cases.r1_plurality_correct)
theta = old_clusters.gain_numerator.sum() / old_clusters.panel_case_denominator.sum()
residual = old_clusters.gain_numerator - theta * old_clusters.panel_case_denominator
g = len(old_clusters)
se = math.sqrt(g / (g - 1) * float((residual ** 2).sum())) / old_clusters.panel_case_denominator.sum()
old_pooled.update({'unique_question_clusters': g, 'panel_case_observations': len(old_cases), 'question_cluster_bootstrap_ci95_pp': bootstrap_cluster_ratio(old_clusters.gain_numerator, old_clusters.panel_case_denominator), 'question_cluster_robust_se_pp': 100 * se, 'question_cluster_t_p': float(2 * student_t.sf(abs(theta / se), g - 1)), 'individual_available_pairs': int(old_clusters.individual_valid_pairs.sum()), 'individual_difference_pp': float(100 * old_clusters.individual_gain_numerator.sum() / old_clusters.individual_valid_pairs.sum()), 'individual_question_cluster_bootstrap_ci95_pp': bootstrap_cluster_ratio(old_clusters.individual_gain_numerator, old_clusters.individual_valid_pairs)})


In [7]:
outputs = {'accuracy': accuracy, 'paired_comparisons': comparisons, 'independent_majority': majorities, 'independent_oracle': oracles, 'rounds': rounds, 'influence': pd.DataFrame(influence_rows), 'legacy_train1000_accuracy': legacy_accuracy, 'overlap': pd.DataFrame(overlap_rows), 'configuration': pd.DataFrame(configuration_rows), 'legacy_pooled_rounds': pd.DataFrame([old_pooled])}
for key, value in outputs.items():
    value.to_csv(OUT / f'{PREFIX}_{key}.csv', index=False, encoding='utf-8-sig')
payload = {'methods': {'correctness': 'Single recorded A-D gold label; invalid predictions counted wrong.', 'paired_test': 'Two-sided exact McNemar, binomial test on discordant paired questions.', 'member_tests_multiplicity': 'Holm within each subset (3 contrasts); additional Holm across all 18 shown.', 'majority_oracle_multiplicity': 'Separate Holm families of 6 subset comparisons for independent majority and oracle.', 'round_tests_multiplicity': 'Separate Holm families: 6 new subsets; 4 legacy panels.', 'r1_vote': 'Plurality of round-1 participant answers; organizer breaks ties, matching NB04.', 'contested': 'At least two rounds executed.', 'individual_intervals': 'Available paired expert decisions only, matching NB04. 10,000 whole-question bootstrap resamples of paired-correctness numerator and available-pair denominator; seed 20261011.', 'accuracy_difference_intervals': '10,000 paired-question bootstrap resamples; seed 20261011; intervals are unadjusted for multiplicity.', 'overlap': 'Never pool the six new overlapping subsets into a grand accuracy or independent paired test.', 'oracle': 'Any independently evaluated member correct; upper bound for selecting existing member answers, not a runnable system.'}, 'source_manifest': manifest['subsets'], 'input_files': input_provenance, **{key: value.astype(object).where(pd.notna(value), None).to_dict(orient='records') for key, value in outputs.items()}}
(OUT / f'{PREFIX}.json').write_text(json.dumps(payload, ensure_ascii=False, indent=2, allow_nan=False), encoding='utf-8')
print('Verified identical case IDs, question wording, options and gold labels inside every comparison.')
print('Accuracy (%):')
print(accuracy.pivot(index='subset', columns='system', values='accuracy_pct').round(3).to_string())
print('Panel-minus-member paired comparisons:')
print(comparisons.round(6).to_string(index=False))
print('Round-1 plurality versus final consensus on contested questions:')
print(rounds[['group', 'panel', 'questions', 'contested', 'accuracy_contested_r1_pct', 'accuracy_contested_final_pct', 'repaired', 'corrupted', 'p_exact', 'p_holm_within_group']].round(6).to_string(index=False))
print('Independent majority and member oracle:')
print(majorities.round(6).to_string(index=False))
print(oracles.round(6).to_string(index=False))
print('All new outputs use the unique prefix', PREFIX)


Verified identical case IDs, question wording, options and gold labels inside every comparison.
Accuracy (%):
system         DeepSeek-V4 Flash  Delphi  GPT-OSS-120B  Qwen3.6-35B-A3B
subset                                                                 
dev4183                   73.655  77.887        72.819           77.480
goa100                    72.000  85.000        77.000           83.000
marc100                   75.000  86.000        78.000           82.000
marc250                   76.400  83.600        76.400           82.000
random_dev300             72.333  77.000        75.333           74.333
shao_hard100              24.000  26.000        27.000           31.000
Panel-minus-member paired comparisons:
       subset        comparator    n  accuracy_a  accuracy_b  difference_pp  a_only_correct  b_only_correct  p_exact  difference_ci_low_pp  difference_ci_high_pp  p_holm_three_members_within_subset  p_holm_all_18
random_dev300      GPT-OSS-120B  300    0.770000    0.753333  

In [8]:
from IPython.display import display
display(accuracy.round(3))
display(comparisons.round(6))
display(rounds.round(4))
display(pd.DataFrame([old_pooled]))
display(pd.DataFrame(influence_rows).round(3))
display(pd.DataFrame(overlap_rows))


,subset,system,n,correct,accuracy_pct,ci_low_pct,ci_high_pct,invalid
0,random_dev300,Delphi,300,231,77.000,71.914,81.403,0
1,random_dev300,GPT-OSS-120B,300,226,75.333,70.155,79.871,0
2,random_dev300,Qwen3.6-35B-A3B,300,223,74.333,69.105,78.947,0
3,random_dev300,DeepSeek-V4 Flash,300,217,72.333,67.013,77.089,0
4,shao_hard100,Delphi,100,26,26.000,18.405,35.371,0
5,shao_hard100,GPT-OSS-120B,100,27,27.000,19.270,36.432,3
6,shao_hard100,Qwen3.6-35B-A3B,100,31,31.000,22.780,40.626,0
7,shao_hard100,DeepSeek-V4 Flash,100,24,24.000,16.691,33.232,0
8,goa100,Delphi,100,85,85.000,76.716,90.694,0
9,goa100,GPT-OSS-120B,100,77,77.000,67.846,84.157,0


,subset,comparator,n,accuracy_a,accuracy_b,difference_pp,a_only_correct,b_only_correct,p_exact,difference_ci_low_pp,difference_ci_high_pp,p_holm_three_members_within_subset,p_holm_all_18
0,random_dev300,GPT-OSS-120B,300,0.770000,0.753333,1.666667,27,22,0.568172,-3.000000,6.333333,0.568172,1.000000
1,random_dev300,Qwen3.6-35B-A3B,300,0.770000,0.743333,2.666667,25,17,0.279956,-1.666667,6.666667,0.559912,1.000000
2,random_dev300,DeepSeek-V4 Flash,300,0.770000,0.723333,4.666667,38,24,0.097954,-0.333333,10.000000,0.293863,0.979544
3,shao_hard100,GPT-OSS-120B,100,0.260000,0.270000,-1.000000,8,9,1.000000,-9.000000,7.000000,1.000000,1.000000
4,shao_hard100,Qwen3.6-35B-A3B,100,0.260000,0.310000,-5.000000,11,16,0.442068,-15.000000,5.000000,1.000000,1.000000
5,shao_hard100,DeepSeek-V4 Flash,100,0.260000,0.240000,2.000000,18,16,0.864166,-9.000000,13.000000,1.000000,1.000000
6,goa100,GPT-OSS-120B,100,0.850000,0.770000,8.000000,9,1,0.021484,2.000000,14.000000,0.042969,0.257812
7,goa100,Qwen3.6-35B-A3B,100,0.850000,0.830000,2.000000,7,5,0.774414,-5.000000,9.000000,0.774414,1.000000
8,goa100,DeepSeek-V4 Flash,100,0.850000,0.720000,13.000000,18,5,0.010622,4.000000,22.000000,0.031866,0.138086
9,marc100,GPT-OSS-120B,100,0.860000,0.780000,8.000000,11,3,0.057373,1.000000,15.000000,0.114746,0.631104


,group,panel,questions,contested,r1_unanimous,stopped_after_r2,ran_r3,mean_rounds,accuracy_all_r1_plurality_pct,accuracy_all_final_pct,...,individual_contested_last_pct,individual_repaired,individual_corrupted,individual_difference_pp,individual_case_bootstrap_ci95_pp,individual_missing_pair_exclusions,invalid_round1_decisions,invalid_last_decisions,winner_consensus_mismatches,p_holm_within_group
0,new,random_dev300,300,104,196,80,24,1.4267,76.6667,77.0000,...,57.0513,63,33,9.6154,"[3.5256410256410255, 15.384615384615385]",0,1,1,0,1.0000
1,new,shao_hard100,100,65,35,38,27,1.9200,28.0000,26.0000,...,24.2268,21,37,-8.2474,"[-16.41025641025641, 0.0]",1,2,2,0,1.0000
2,new,goa100,100,24,76,21,3,1.2700,85.0000,85.0000,...,66.6667,16,7,12.5000,"[0.0, 23.61111111111111]",0,0,0,0,1.0000
3,new,marc100,100,27,73,20,7,1.3400,82.0000,86.0000,...,68.7500,23,2,26.2500,"[16.25, 35.80246913580247]",1,1,0,0,0.7500
4,new,marc250,250,62,188,44,18,1.3200,83.2000,83.6000,...,64.1304,44,15,15.7609,"[7.608695652173914, 23.655913978494624]",2,2,0,0,1.0000
5,new,dev4183,4183,1231,2952,960,271,1.3591,77.9106,77.8867,...,51.1678,696,474,6.0293,"[4.097637781946101, 7.926167209554831]",11,30,25,0,1.0000
6,legacy_train1000,DE-7G,1000,173,827,108,65,1.2380,87.0000,88.1000,...,68.2875,119,33,18.1818,"[13.49036402569593, 22.594142259414227]",46,119,89,0,0.1229
7,legacy_train1000,DE-5G,1000,179,821,129,50,1.2290,86.5000,87.6000,...,65.1639,117,49,13.9344,"[8.939339948467738, 18.775819418676555]",49,222,187,0,0.1041
8,legacy_train1000,DE-6G,1000,199,801,124,75,1.2740,85.1000,86.2000,...,59.0444,133,59,12.6280,"[7.705479452054795, 17.406691960753733]",11,17,18,0,0.1229
9,legacy_train1000,DH-D,1000,82,918,55,27,1.1090,87.0000,89.3000,...,59.2000,32,16,12.8000,"[0.0, 24.324911868390124]",121,310,222,0,0.0005


,n,accuracy_a,accuracy_b,difference_pp,a_only_correct,b_only_correct,p_exact,unique_question_clusters,panel_case_observations,question_cluster_bootstrap_ci95_pp,question_cluster_robust_se_pp,question_cluster_t_p,individual_available_pairs,individual_difference_pp,individual_question_cluster_bootstrap_ci95_pp
0,633,0.624013,0.535545,8.846761,90,34,5.088330e-07,297,633,"[5.047119898656201, 12.581699346405228]",1.930385,0.000007,1672,14.593301,"[11.428526334776334, 17.619648677436274]"


,group,panel,position,majority_correct,n_decisions,ended_on_initial_majority,share_pct
0,new,random_dev300,majority,False,86,76,88.372
1,new,random_dev300,majority,True,110,106,96.364
2,new,random_dev300,minority,False,43,36,83.721
3,new,random_dev300,minority,True,55,53,96.364
4,new,shao_hard100,majority,False,72,63,87.500
5,new,shao_hard100,majority,True,28,20,71.429
6,new,shao_hard100,minority,False,36,31,86.111
7,new,shao_hard100,minority,True,14,11,78.571
8,new,goa100,majority,False,12,12,100.000
9,new,goa100,majority,True,32,32,100.000


,subset_a,subset_b,shared_question_ids
0,random_dev300,shao_hard100,8
1,random_dev300,goa100,8
2,random_dev300,marc100,0
3,random_dev300,marc250,0
4,random_dev300,dev4183,300
5,shao_hard100,goa100,0
6,shao_hard100,marc100,0
7,shao_hard100,marc250,0
8,shao_hard100,dev4183,100
9,goa100,marc100,0
